# Modul 1 — Django Fundamentals

**Studi kasus modul ini & Modul 2:** aplikasi To-Do List sederhana.

## Tujuan Pembelajaran
1. Menjelaskan arsitektur MVT (Model-View-Template) pada Django.
2. Membuat project dan app Django dari command line.
3. Membuat routing URL dengan `urls.py`.
4. Membuat *function-based view* dan mengembalikan response.
5. Membuat template HTML dan merender data dari view ke template.
6. Menjalankan development server dan mengujinya lewat browser.

> **Cara pakai notebook ini:** notebook ini adalah **panduan referensi**, bukan sesuatu yang dijalankan cell-per-cell. Semua command dijalankan di terminal/CMD kamu sendiri, dan semua isi file dicontohkan di sini untuk kamu ketik/salin ke editor. Ikuti urutannya dari atas ke bawah.


## 1. Apa itu Django?

Django adalah web framework Python yang mengikuti pola **MVT (Model-View-Template)**:

| Komponen | Peran |
|---|---|
| **Model** | Struktur & akses data (database) |
| **View** | Logika: memproses request, menyiapkan data |
| **Template** | Tampilan (HTML) yang dikirim ke browser |

Alur: **Request masuk → dicocokkan ke `urls.py` → dijalankan sebuah View → View menyiapkan data & memilih Template → Template dirender jadi HTML → Response dikirim ke browser.**

Modul 1 fokus ke jalur **urls → views → templates** dengan data statis dulu. Modul 2 mengganti data statis itu dengan **Model** dari database.


## 2. Instalasi Django

Jalankan di terminal:

```bash
pip install django
```

Cek instalasi berhasil:

```bash
python -m django --version
```


## 3. Membuat Folder Kerja & Project

```bash
mkdir proyek-todo-app
cd proyek-todo-app
django-admin startproject todo_project .
```

Syntax: `django-admin startproject <nama_project> <lokasi>`. Tanda titik `.` berarti "buat langsung di folder saat ini" (kalau titik dihilangkan, Django akan bikin folder bersarang tambahan bernama `todo_project`).


## 4. Anatomi Project Django

Setelah `startproject`, struktur foldernya:

```
proyek-todo-app/
├── manage.py
└── todo_project/
    ├── __init__.py
    ├── settings.py
    ├── urls.py
    ├── asgi.py
    └── wsgi.py
```

- **`manage.py`** — command-line utility: menjalankan server, migration, dan management command lain.
- **`todo_project/settings.py`** — konfigurasi project: `INSTALLED_APPS`, database, template, dsb.
- **`todo_project/urls.py`** — "peta jalan" utama: memetakan URL ke app/view mana yang menangani.
- **`todo_project/asgi.py` & `wsgi.py`** — entry point untuk deployment (server production).

Project (`todo_project`) itu wadah konfigurasi. Fitur aplikasi sesungguhnya (to-do list, blog, dst) kita taruh di dalam **app**, yang bisa lebih dari satu dalam satu project.


## 5. Membuat App `tasks`

```bash
python manage.py startapp tasks
```

Syntax: `python manage.py startapp <nama_app>`. Satu project Django bisa berisi banyak app.


## 6. Mendaftarkan App ke `INSTALLED_APPS`

App yang baru dibuat **tidak otomatis aktif**. Buka `todo_project/settings.py`, cari list `INSTALLED_APPS`, lalu tambahkan `'tasks'`:

```python
INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
    'tasks',            # <-- baris baru
]
```


## 7. URL Routing

Pola dasar routing Django:
```python
path(route, view, name="nama_url")
```
Praktik umum: tiap app punya `urls.py` sendiri, di-*include* dari `urls.py` project utama.

Buat file **`tasks/urls.py`**:

```python
from django.urls import path
from . import views

urlpatterns = [
    path('', views.index, name='index'),
]
```

Lalu edit **`todo_project/urls.py`** supaya meng-*include* `tasks.urls`:

```python
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('tasks.urls')),
]
```


## 8. Views — Function-Based View

View = fungsi Python yang menerima `request`, mengembalikan `response` (biasanya lewat `render()`).

Untuk Modul 1, data tugas masih **hardcode** di dalam view (belum dari database — itu tugas Modul 2). Isi **`tasks/views.py`**:

```python
from django.shortcuts import render


def index(request):
    # Data masih hardcode di sini.
    # Modul 2 akan mengganti ini dengan query ke database lewat model Task.
    tasks = [
        {"title": "Belajar dasar Django", "is_done": True},
        {"title": "Membuat app pertama", "is_done": True},
        {"title": "Menghubungkan ke database", "is_done": False},
        {"title": "Membuat form input tugas", "is_done": False},
    ]
    context = {"tasks": tasks, "total": len(tasks)}
    return render(request, "tasks/index.html", context)
```


## 9. Templates

Template = file HTML dengan *template tags*: `{{ variabel }}` untuk menampilkan nilai, `{% tag %}` untuk logika (perulangan/percabangan).

Django mencari template di folder `templates/<nama_app>/` di dalam masing-masing app secara default. Buat foldernya:

```bash
mkdir -p tasks/templates/tasks
```

Lalu buat file **`tasks/templates/tasks/index.html`**:

```html
<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <title>To-Do List</title>
</head>
<body>
    <h1>Daftar Tugas Saya</h1>
    <p>Total tugas: {{ total }}</p>
    <ul>
        {% for task in tasks %}
            <li>
                {% if task.is_done %}
                    <s>{{ task.title }}</s> (selesai)
                {% else %}
                    {{ task.title }}
                {% endif %}
            </li>
        {% empty %}
            <li>Belum ada tugas.</li>
        {% endfor %}
    </ul>
</body>
</html>
```


## 10. Menjalankan Server & Menguji di Browser

```bash
python manage.py runserver
```

Output yang muncul kira-kira:
```
Starting development server at http://127.0.0.1:8000/
Quit the server with CTRL-BREAK.
```

Buka browser ke `http://127.0.0.1:8000/` — kamu akan melihat halaman "Daftar Tugas Saya" dengan 4 item (2 dicoret karena `is_done=True`). Untuk berhenti, tekan `CTRL+C` (atau `CTRL+BREAK` di Windows) di terminal.


## 11. Preview: Django Admin

Django menyediakan panel admin siap pakai di `/admin/` untuk mengelola data lewat UI, tanpa perlu bikin dashboard sendiri. Panel ini baru berguna setelah kita punya **model** (tabel database) untuk didaftarkan — itu akan kita lakukan di **Modul 2**, sekaligus membuat superuser dan login ke admin.


## 12. Ringkasan

- Django memakai pola **MVT**: urls → views → templates.
- **Project** = wadah konfigurasi, **App** = unit fitur (bisa banyak app per project).
- Command inti: `django-admin startproject`, `python manage.py startapp`, `python manage.py runserver`.
- **View** = fungsi Python yang mengembalikan response, biasanya lewat `render()`.
- **Template** = HTML + template tags (`{{ }}`, `{% %}`) untuk menampilkan data dari view.

Di Modul 2, data hardcode di `views.py` akan diganti dengan **Model** yang tersimpan di database, lengkap dengan ORM untuk CRUD dan **Form** untuk input data dari user.


## 13. Exercise (dikerjakan saat sesi kelas)

Tambahkan halaman **"About"** yang menampilkan info singkat tentang aplikasi To-Do List ini.

1. Tambahkan view `about` di `tasks/views.py` yang me-render template `tasks/about.html` (judul + satu paragraf teks).
2. Tambahkan route `about/` di `tasks/urls.py`, dengan `name='about'`.
3. Buat file `tasks/templates/tasks/about.html`.
4. Jalankan `python manage.py runserver` dan buka `http://127.0.0.1:8000/about/` di browser untuk memastikan halamannya muncul.


## 14. Homework (dikerjakan di luar sesi kelas)

Kembangkan aplikasi To-Do List ini dengan menambahkan:

1. View **`contact`** (`/contact/`) — info kontak pengajar/kursus (boleh data dummy).
2. View **`stats`** (`/stats/`) — hitung dari list tugas hardcode di `views.py`: jumlah selesai vs belum selesai, tampilkan sebagai teks (contoh: "2 dari 4 tugas selesai").
3. Pakai **template inheritance**: buat `tasks/templates/tasks/base.html` (struktur HTML dasar + nav sederhana), lalu ubah `index.html`, `about.html`, `contact.html`, `stats.html` supaya memakai `{% extends "tasks/base.html" %}` dan `{% block content %}`.

**Kriteria selesai:** keempat halaman (`index`, `about`, `contact`, `stats`) bisa dibuka lewat browser (`python manage.py runserver`) tanpa error, dan semuanya memakai `base.html` yang sama.
